# 第10章：算子融合 — 减少内存访问的关键技术

## 前置知识
- 第01-09章

## 学习目标
- 理解为什么算子融合（Kernel Fusion）是 GPU 优化的关键
- 实现 MatMul + Bias + ReLU 的融合 kernel
- 实现融合的 LayerNorm kernel
- 掌握 Triton 相比 PyTorch 在融合方面的优势

In [ ]:
import torch
import triton
import triton.language as tl

## 10.1 为什么需要算子融合？

在 PyTorch 中，每个操作都是一个独立的 kernel：

```
PyTorch 的执行方式（3 个 kernel）:

Kernel 1: MatMul           Kernel 2: Bias Add        Kernel 3: ReLU
┌─────────────────┐        ┌─────────────────┐       ┌─────────────────┐
│ 读取 A, B       │        │ 读取 C, bias     │       │ 读取 C'         │
│ 计算 C = A @ B  │ ──→    │ 计算 C' = C+bias │ ──→   │ 计算 C''=ReLU(C')│
│ 写入 C 到 DRAM  │  DRAM  │ 写入 C' 到 DRAM │ DRAM  │ 写入 C'' 到 DRAM│
└─────────────────┘        └─────────────────┘       └─────────────────┘
                    ↑                          ↑
                 额外的 DRAM 读写！          又一次 DRAM 读写！

融合版本（1 个 kernel）:
┌──────────────────────────────────┐
│ 读取 A, B, bias                  │
│ 计算 C = ReLU(A @ B + bias)     │  ← 中间结果只在寄存器/共享内存中
│ 写入 C 到 DRAM                   │
└──────────────────────────────────┘
```

### 性能影响

对于 M=N=2048 的 FP16 矩阵：
- 每次 DRAM 读写: 2048 × 2048 × 2 bytes = **8 MB**
- 3 个 kernel 额外的 DRAM 访问: **~32 MB**
- GPU DRAM 带宽约 600 GB/s → 额外开销 ~50μs

对于计算密集的 GEMM，这个开销占比不大；
但对于 element-wise 操作（LayerNorm、Activation），**内存访问是主要瓶颈**！

## 10.2 融合 MatMul + Bias + ReLU

In [ ]:
@triton.jit
def matmul_bias_relu_kernel(
    # 矩阵指针
    a_ptr, b_ptr, bias_ptr, c_ptr,
    # 维度
    M, N, K,
    # 步长
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    # 块大小
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    # 激活函数选择
    ACTIVATION: tl.constexpr,
):
    """
    融合的 MatMul + Bias + Activation:
    C = activation(A @ B + bias)
    
    关键：bias 和 activation 在 GEMM 的最后一步直接应用，
    中间结果不需要写回 DRAM。
    """
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    # 累加器 (FP32)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    # K 维循环: 标准分块 GEMM
    for k in range(0, K, BLOCK_K):
        # 加载 A 的子块 (BLOCK_M x BLOCK_K)
        a_offsets_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        a_offsets_k = k + tl.arange(0, BLOCK_K)
        a_ptrs = a_ptr + a_offsets_m[:, None] * stride_am + a_offsets_k[None, :] * stride_ak
        a_mask = (a_offsets_m[:, None] < M) & (a_offsets_k[None, :] < K)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        
        # 加载 B 的子块 (BLOCK_K x BLOCK_N)
        b_offsets_k = k + tl.arange(0, BLOCK_K)
        b_offsets_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        b_ptrs = b_ptr + b_offsets_k[:, None] * stride_bk + b_offsets_n[None, :] * stride_bn
        b_mask = (b_offsets_k[:, None] < K) & (b_offsets_n[None, :] < N)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        
        # 块级矩阵乘并累加
        acc += tl.dot(a, b)
    
    # ===== 融合部分开始 =====
    # Bias 加法: acc 在寄存器中，不需要写回再读取
    bias_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    bias_mask = bias_offsets < N
    bias = tl.load(bias_ptr + bias_offsets, mask=bias_mask, other=0.0)
    acc += bias[None, :]  # 广播: (1, BLOCK_N) → (BLOCK_M, BLOCK_N)
    
    # 激活函数: 同样在寄存器中直接操作
    if ACTIVATION == "relu":
        acc = tl.maximum(acc, 0.0)
    elif ACTIVATION == "gelu":
        acc = acc * tl.sigmoid(1.702 * acc)
    elif ACTIVATION == "silu":
        acc = acc * tl.sigmoid(acc)
    # ===== 融合部分结束 =====
    
    # 只在最后写入一次 DRAM
    c_offsets_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    c_offsets_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + c_offsets_m[:, None] * stride_cm + c_offsets_n[None, :] * stride_cn
    c_mask = (c_offsets_m[:, None] < M) & (c_offsets_n[None, :] < N)
    tl.store(c_ptrs, acc.to(tl.float16), mask=c_mask)

In [ ]:
def matmul_bias_activation(a, b, bias, activation="relu"):
    """融合的 MatMul + Bias + Activation"""
    assert a.shape[1] == b.shape[0]
    M, K = a.shape
    K, N = b.shape
    c = torch.empty(M, N, device=a.device, dtype=torch.float16)
    
    BLOCK_M, BLOCK_N, BLOCK_K = 64, 64, 32
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    
    matmul_bias_relu_kernel[grid](
        a, b, bias, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        ACTIVATION=activation,
    )
    return c

# 测试
M, N, K = 512, 512, 256
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)
bias = torch.randn(N, device='cuda', dtype=torch.float16)

# Triton 融合版
c_triton = matmul_bias_activation(a, b, bias, "relu")

# PyTorch 分步版
c_torch = torch.relu(a.float() @ b.float() + bias.float()).half()

print(f"MatMul+Bias+ReLU 融合:")
print(f"  形状: ({M},{K}) @ ({K},{N}) + ({N},) → ({M},{N})")
print(f"  正确性: {torch.allclose(c_triton, c_torch, atol=1e-1)}")
print(f"  最大误差: {(c_triton.float() - c_torch.float()).abs().max().item():.4f}")

## 10.3 融合 LayerNorm

LayerNorm 是 Transformer 的核心组件，公式：

```
LayerNorm(x) = γ * (x - μ) / √(σ² + ε) + β

其中:
  μ = mean(x)           (沿最后一维)
  σ² = var(x)           (沿最后一维)
  γ, β = 可学习参数
```

在 PyTorch 中，`F.layer_norm` 需要多次全局内存读写：
```
1. 读 x → 计算 mean → 写 mean
2. 读 x, mean → 计算 var → 写 var
3. 读 x, mean, var, γ, β → 归一化 → 写输出
```

融合版本: 一次读取 x，在寄存器中完成所有计算：

In [ ]:
@triton.jit
def layernorm_kernel(
    x_ptr, gamma_ptr, beta_ptr, out_ptr,
    N,  # 归一化维度大小
    stride,  # 行步长
    eps,
    BLOCK_N: tl.constexpr,
):
    """
    融合 LayerNorm: 一个 kernel 完成 mean、var、normalize 全部计算。
    每个 program 处理一行。
    """
    row_idx = tl.program_id(0)
    
    # 加载一整行
    col_offsets = tl.arange(0, BLOCK_N)
    mask = col_offsets < N
    x = tl.load(x_ptr + row_idx * stride + col_offsets, mask=mask, other=0.0).to(tl.float32)
    
    # === 在寄存器中完成所有计算 ===
    
    # 1. 计算均值
    mean = tl.sum(x, axis=0) / N
    
    # 2. 计算方差
    diff = tl.where(mask, x - mean, 0.0)
    var = tl.sum(diff * diff, axis=0) / N
    
    # 3. 归一化
    inv_std = 1.0 / tl.sqrt(var + eps)
    normalized = diff * inv_std
    
    # 4. 缩放和偏移 (γ * normalized + β)
    gamma = tl.load(gamma_ptr + col_offsets, mask=mask, other=1.0).to(tl.float32)
    beta = tl.load(beta_ptr + col_offsets, mask=mask, other=0.0).to(tl.float32)
    out = gamma * normalized + beta
    
    # 写入输出
    tl.store(out_ptr + row_idx * stride + col_offsets, out.to(tl.float16), mask=mask)

In [ ]:
def layernorm(x, gamma, beta, eps=1e-5):
    """融合 LayerNorm 包装函数"""
    assert x.ndim == 2
    M, N = x.shape
    out = torch.empty_like(x)
    
    BLOCK_N = triton.next_power_of_2(N)
    layernorm_kernel[(M,)](
        x, gamma, beta, out,
        N, x.stride(0), eps,
        BLOCK_N=BLOCK_N,
    )
    return out

# 测试
M, N = 256, 768  # 典型的 Transformer 维度
x = torch.randn(M, N, device='cuda', dtype=torch.float16)
gamma = torch.ones(N, device='cuda', dtype=torch.float16)
beta = torch.zeros(N, device='cuda', dtype=torch.float16)

out_triton = layernorm(x, gamma, beta)
out_torch = torch.nn.functional.layer_norm(x.float(), (N,), gamma.float(), beta.float()).half()

print(f"LayerNorm 融合:")
print(f"  形状: ({M}, {N})")
print(f"  正确性: {torch.allclose(out_triton, out_torch, atol=5e-2)}")
print(f"  最大误差: {(out_triton.float() - out_torch.float()).abs().max().item():.6f}")

## 10.4 性能对比

In [ ]:
# LayerNorm 性能对比
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['N'],
        x_vals=[128, 256, 512, 768, 1024, 2048, 4096],
        line_arg='provider',
        line_vals=['triton', 'torch'],
        line_names=['Triton (fused)', 'PyTorch'],
        styles=[('blue', '-'), ('green', '-')],
        ylabel='GB/s',
        plot_name='LayerNorm 性能对比 (M=4096)',
        args={'M': 4096},
    )
)
def bench_layernorm(M, N, provider):
    x = torch.randn(M, N, device='cuda', dtype=torch.float16)
    gamma = torch.ones(N, device='cuda', dtype=torch.float16)
    beta = torch.zeros(N, device='cuda', dtype=torch.float16)
    quantiles = [0.5, 0.2, 0.8]
    if provider == 'triton':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: layernorm(x, gamma, beta), quantiles=quantiles
        )
    else:
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: torch.nn.functional.layer_norm(x.float(), (N,), gamma.float(), beta.float()),
            quantiles=quantiles
        )
    gbps = lambda ms: 2 * M * N * x.element_size() / ms * 1e-6
    return gbps(ms), gbps(max_ms), gbps(min_ms)

bench_layernorm.run(print_data=True, show_plots=True)

## 10.5 融合的威力总结

```
操作类型           融合效果
─────────────────────────────────────────────
ElementWise 链     ★★★★★  效果最好（全部省掉中间 DRAM 访问）
(ReLU, GELU, Add)  

Reduction + EW     ★★★★☆  LayerNorm, Softmax
(mean, var, norm)

GEMM + EW          ★★★☆☆  MatMul + Bias + Act
(后处理融合)        中间矩阵不写 DRAM

GEMM + GEMM        ★★☆☆☆  Attention 中的两个 MatMul
(FlashAttention)   复杂但回报巨大
```

**Triton 的核心优势**: 写融合 kernel 就像写普通 Python，不需要像 CUDA 那样手动管理共享内存和线程同步。

## 练习

1. **Fused Dropout + LayerNorm**: 在 LayerNorm kernel 中加入 dropout（提示：使用 `tl.rand` 生成随机数）
2. **Fused Softmax + Scale**: 实现 `softmax(x / temperature)` 的融合版本
3. **RMSNorm**: 实现融合的 RMSNorm（LLaMA 使用的归一化）：`x * γ / √(mean(x²) + ε)`
4. **GEMM + SiLU + GEMM**: 在 SwiGLU FFN 中，计算 `(xW₁ * SiLU(xW₃)) @ W₂`